# G3b · Second backbone: TANL and ReNeg with CLIP ViT-L/14 (optional, after G3)

The same evaluation as G3 on the ViT-L/14 cache (built on Kaggle, step 11; P2 already copied it to Drive): TANL
and ReNeg-balanced with the blind and the full pool, three stream orders, both protocols. Nothing is tuned: the
settings are the ViT-B/16 ones. The knowledge-graph names are encoded here with the ViT-L/14 text encoder
(same prompt as the text bank). A protocol whose sets are missing from the cache is skipped with a message.

**Runtime:** GPU needed; a check cell as in G3, then roughly 1–3 hours on a T4. Saved after every method and
protocol: after a dropped session, run all cells again. **Output:** `MyDrive/ReNeg/transfers/g3_final_l14/`
`g3_summary.csv`, `g3_runs.csv` and `g3_check.json`, as in results/g3b_vitl14/.

In [ ]:
# ── Colab set-up (the same cell in every ReNeg notebook) ─────────────────────────────
import os, sys
DRIVE_ROOT = "/content/drive/MyDrive/ReNeg"          # change only if your project folder is elsewhere
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:                                    # running outside Colab (tests)
    IN_COLAB = False
    DRIVE_ROOT = os.environ.get("RENEG_DRIVE_ROOT", os.path.abspath("ReNeg"))
HOME = os.environ.get("RENEG_OODLAB_HOME", "/content/oodlab_home" if IN_COLAB else os.path.abspath("oodlab_home"))
for d in (f"{DRIVE_ROOT}/oodlab_working", f"{HOME}/scratch", f"{HOME}/input"):
    os.makedirs(d, exist_ok=True)
if not os.path.islink(f"{HOME}/working"):              # working/ lives on Drive, scratch/ stays local
    if os.path.isdir(f"{HOME}/working") and not os.listdir(f"{HOME}/working"):
        os.rmdir(f"{HOME}/working")
    if not os.path.exists(f"{HOME}/working"):
        os.symlink(f"{DRIVE_ROOT}/oodlab_working", f"{HOME}/working")
os.environ["OODLAB_HOME"] = HOME
# a newer ReNeg_Colab_bundle*.zip in uploads/ is installed here, so a new bundle needs no C0 re-run
import glob, re, shutil, zipfile
def _reneg_version(text):
    m = re.search(r'__version__ = "([0-9.]+)"', text or "")
    return tuple(int(x) for x in m.group(1).split(".")) if m else None
_init = f"{DRIVE_ROOT}/code/reneg_code/reneg/__init__.py"
_installed = _reneg_version(open(_init).read()) if os.path.isfile(_init) else None
_newest = None
for _zp in glob.glob(f"{DRIVE_ROOT}/uploads/*.zip"):
    try:
        with zipfile.ZipFile(_zp) as _z:
            _hit = [n for n in _z.namelist() if n.endswith("reneg_code/reneg/__init__.py")]
            _v = _reneg_version(_z.read(_hit[0]).decode()) if _hit else None
        if _v and (_newest is None or _v > _newest[0]):
            _newest = (_v, _zp, _hit[0][: -len("reneg_code/reneg/__init__.py")])
    except (zipfile.BadZipFile, OSError):
        pass
if _newest and (_installed is None or _newest[0] > _installed):
    _tmp = os.path.join(HOME, "scratch", "_bundle")
    shutil.rmtree(_tmp, ignore_errors=True)
    with zipfile.ZipFile(_newest[1]) as _z:
        _z.extractall(_tmp)
    _src = os.path.join(_tmp, _newest[2])
    shutil.rmtree(f"{DRIVE_ROOT}/code/reneg_code", ignore_errors=True)
    shutil.copytree(os.path.join(_src, "reneg_code"), f"{DRIVE_ROOT}/code/reneg_code")
    os.makedirs(f"{DRIVE_ROOT}/notebooks", exist_ok=True)
    for _f in os.listdir(os.path.join(_src, "notebooks")):
        if _f.endswith(".ipynb") and not _f.startswith("C0"):
            shutil.copy2(os.path.join(_src, "notebooks", _f), f"{DRIVE_ROOT}/notebooks/{_f}")
    _fmt = lambda v: ".".join(map(str, v)) if v else "none"
    print(f"reneg code updated {_fmt(_installed)} -> {_fmt(_newest[0])} from uploads/{os.path.basename(_newest[1])}")
    if "reneg" in sys.modules:
        print("reneg was already imported in this session: Runtime -> Restart session, then run again")
CODE_ROOT = f"{DRIVE_ROOT}/code/oodlab_code"           # same name as in the Kaggle notebooks
for p in (CODE_ROOT, f"{DRIVE_ROOT}/code/reneg_code"):
    if not os.path.isdir(p):
        raise FileNotFoundError(f"{p} not found: run C0_transfer_from_kaggle first (GUIDE_COLAB.md, step 4)")
    if p not in sys.path:
        sys.path.insert(0, p)
import oodlab, reneg
print(f"oodlab {oodlab.__version__} | reneg {reneg.__version__} | project folder {DRIVE_ROOT}")

In [ ]:
from oodlab.session import start, finish, ensure_packages
ensure_packages(clip=True)
ctx = start("G3b_final_eval_vitl14", need_gpu=False)
log = ctx.log
import json, time
import numpy as np, pandas as pd, torch
from oodlab.report import find_cache
from oodlab.runner import Runner
from oodlab.methods import TANL, TANLConfig
from reneg.core import ReNegConfig
from reneg.packs import load_kg_pool
from reneg.reneg_tanl import ReNegTANL
from reneg.transport import l2n
BACKBONE = "ViT-L/14"
cache = find_cache(ctx, BACKBONE)
bank = cache.load_textbank()
OUT = f"{DRIVE_ROOT}/transfers/g3_final_l14"
os.makedirs(OUT, exist_ok=True)
SEEDS = [0, 1, 2]
runner = Runner(cache, device=ctx.device, logger=log)
PROTOCOLS = []
for p in ("openood_v15", "four_ood"):
    if runner.missing(p):
        print(f"{p}: skipped, sets missing from the {BACKBONE} cache: {runner.missing(p)}")
    else:
        PROTOCOLS.append(p)
print(BACKBONE, "| device:", ctx.device, "| sets:", cache.available(), "| protocols:", PROTOCOLS)

## The knowledge-graph pool and the ReNeg settings

In [ ]:
kg = load_kg_pool()["pool"]
pool_file = os.path.join(OUT, "pool_text_vitl14.npz")
names = [r["name"] for r in kg]
if os.path.isfile(pool_file) and list(np.load(pool_file)["pool_names"]) == names:
    pool_text = l2n(torch.tensor(np.load(pool_file)["pool_text"]).float())
else:
    from oodlab.clipwrap import load_clip
    from reneg.g1 import clip_text_encoder
    WEIGHTS = f"{DRIVE_ROOT}/weights"
    model = load_clip(BACKBONE, device=ctx.device, input_roots=ctx.input_roots + [WEIGHTS], download_root=WEIGHTS,
                      random_init=os.environ.get("RENEG_FAKE_GEN") == "1")
    enc = clip_text_encoder(model, prompt="The nice {}.")
    ok = float((enc(list(bank.id_names[:20])).float() * l2n(bank.id_text[:20].float().cpu())).sum(1).min())
    print(f"encoder reproduces the bank's ID embeddings: min cosine {ok:.6f} (should be above 0.999)")
    pool_text = l2n(torch.cat([enc(names[s:s + 2000]).float().cpu() for s in range(0, len(names), 2000)]))
    np.savez_compressed(pool_file, pool_text=pool_text.half().numpy(), pool_names=np.asarray(names))
    del model
pool_cluster = np.array([r["cluster"] for r in kg])
blind = ~np.array([bool(r["in_ssb_hard"]) or bool(r["in_ninco"]) for r in kg])
POOLS = {"full": np.arange(len(kg)), "blind": np.nonzero(blind)[0]}
BASE = dict(n_min=1, pool_frac=0.0, pool_in_text=False, id_admit="softmax+base")
CONFIGS = {"balanced": ReNegConfig(**BASE, image_mode="kg_clip", vote=True),
           "max": ReNegConfig(**BASE),
           "safe": ReNegConfig(**BASE, image_mode="kg_clip_caught", vote=True)}
RUNS = [("tanl", None, None)] + [("balanced", "balanced", p) for p in ("blind", "full")]
if not hasattr(TANL, "selected_T"):                       # the offline test stub of oodlab, not the real one
    RUNS = RUNS[:1]
    print("oodlab test stub: only the TANL row runs")
print(len(kg), "KG names;", int(blind.sum()), "in the blind pool;", tuple(pool_text.shape))

def make(name, cfg_name, pool, reneg_device=None, id_prior=None):
    if pool is None:
        return TANL(bank.id_text, bank.corpus_text, bank.noise_feats, TANLConfig.paper(),
                    device=ctx.device, n_neglabel=int(bank.n_selected))
    idx = torch.as_tensor(POOLS[pool])
    return ReNegTANL(bank.id_text, bank.corpus_text, bank.noise_feats, pool_text=pool_text[idx],
                     pool_cluster=pool_cluster[POOLS[pool]], cfg=TANLConfig.paper(), rcfg=CONFIGS[cfg_name],
                     device=ctx.device, n_neglabel=int(bank.n_selected), reneg_device=reneg_device,
                     id_prior=id_prior)

runs_path = os.path.join(OUT, "g3_runs.csv")
check_path = os.path.join(OUT, "g3_check.json")

## Check: ReNeg on the GPU gives the same result as on the CPU (about 5 minutes)

TANL always runs on the GPU. ReNeg's own part (the knowledge-graph gate and the image prototypes) can run on
the GPU too, which is much faster. This cell runs ReNeg-balanced on NINCO (one stream order) both ways and
keeps the GPU only when both give the same FPR95 and AUROC (within float rounding). Otherwise everything still
runs, with ReNeg's part on the CPU (slower). The decision is saved, so a re-run skips this check.

In [ ]:
RENEG_DEVICE = ctx.device
if os.path.isfile(check_path):
    chk = json.load(open(check_path))
    RENEG_DEVICE = chk["reneg_device"]
    print("check done already:", chk)
elif ctx.device == "cpu" or len(RUNS) == 1 or "openood_v15" not in PROTOCOLS:
    print("no GPU (or no ReNeg rows): nothing to check")
else:
    res = {}
    for dev in (ctx.device, "cpu"):
        t0 = time.time()
        try:
            d = runner.evaluate(make("balanced", "balanced", "full", reneg_device=dev), "openood_v15",
                                seeds=[0], datasets=["ninco"], config=f"check_{dev}")
            res[dev] = {"fpr95": round(float(d.fpr95.iloc[0]), 3), "auroc": round(float(d.auroc.iloc[0]), 3),
                        "img_per_s": float(d.img_per_s.iloc[0]), "minutes": round((time.time() - t0) / 60, 1)}
        except Exception as e:                                   # noqa: BLE001
            res[dev] = {"error": f"{type(e).__name__}: {e}"}
        print(dev, res[dev])
    g, c = res[ctx.device], res["cpu"]
    if "error" in c and "error" in g:
        raise RuntimeError(f"ReNeg fails on both devices: {res} - please report this output")
    same = ("error" not in g and "error" not in c and abs(g["fpr95"] - c["fpr95"]) <= 0.5
            and abs(g["auroc"] - c["auroc"]) <= 0.2)
    RENEG_DEVICE = ctx.device if same else "cpu"
    chk = {"reneg_device": RENEG_DEVICE, "results": res}
    json.dump(chk, open(check_path, "w"), indent=1)
    print("ReNeg runs on:", RENEG_DEVICE, "(GPU and CPU agree)" if same else "(GPU result differs or failed: CPU)")
if os.path.isfile(check_path):              # time estimate (upper bound: TANL alone is faster)
    sets = {"openood_v15": (["ssb_hard", "ninco", "inaturalist", "textures", "openimage_o"], "imagenet_test"),
            "four_ood": (["inaturalist", "sun", "places", "textures_all"], "imagenet_val_all")}
    n_img = sum(sum(len(runner.get(n)) for n in sets[p][0]) + len(sets[p][0]) * len(runner.get(sets[p][1]))
                for p in PROTOCOLS)
    speed = json.load(open(check_path))["results"][RENEG_DEVICE]["img_per_s"]
    print(f"estimate: {len(RUNS)} methods x {len(SEEDS)} orders x {n_img:,} images at about {speed:.0f} img/s "
          f"= {len(RUNS) * len(SEEDS) * n_img / speed / 3600:.1f} hours at most")

## Run (saved after every method and protocol; re-running skips what is done)

In [ ]:
done = pd.read_csv(runs_path) if os.path.isfile(runs_path) else pd.DataFrame()
for name, cfg_name, pool in RUNS:
    label = name if pool is None else f"reneg_{cfg_name}_{pool}"
    for protocol in PROTOCOLS:
        if len(done) and ((done["label"] == label) & (done["protocol"] == protocol)).any():
            print("done already:", label, protocol)
            continue
        t0 = time.time()
        method = make(name, cfg_name, pool, reneg_device=RENEG_DEVICE)
        df = runner.evaluate(method, protocol, seeds=SEEDS, config=label)
        df["label"] = label
        df["backbone"] = BACKBONE
        done = pd.concat([done, df], ignore_index=True)
        done.to_csv(runs_path, index=False)
        print(f"{label} {protocol}: {(time.time() - t0) / 60:.1f} min")

## Summary (mean over the three stream orders), then attach the three files

In [ ]:
df = pd.read_csv(runs_path)
per = df.groupby(["label", "protocol", "group", "dataset"])[["fpr95", "auroc"]].mean().reset_index()
rows = []
for label, g in per.groupby("label"):
    row = {"method": label}
    for (protocol, group), gg in g.groupby(["protocol", "group"]):
        row[f"{group} FPR95"] = round(gg["fpr95"].mean(), 2)
        row[f"{group} AUROC"] = round(gg["auroc"].mean(), 2)
    for _, r in g.iterrows():                              # iNaturalist is in several protocols: prefix them
        pre = {"openood_v15": "", "four_ood": "4ood_", "four_ood_45k": "4ood45_"}.get(r["protocol"], r["protocol"] + "_")
        row[pre + r["dataset"]] = round(r["fpr95"], 2)
    rows.append(row)
summary = pd.DataFrame(rows)
summary.to_csv(os.path.join(OUT, "g3_summary.csv"), index=False)
pd.set_option("display.width", 250)
print(BACKBONE)
print(summary.to_string(index=False))
print("\nresult files:")
for f in (runs_path, os.path.join(OUT, "g3_summary.csv"), check_path):
    if os.path.isfile(f):
        print("  ", f)
finish(ctx, {"runs": int(len(df)), "labels": sorted(df["label"].unique().tolist()), "reneg_device": RENEG_DEVICE,
             "backbone": BACKBONE})

### Finish saving to Google Drive

Colab copies files to Drive in the background. This cell waits until every file is written, so run it before
you disconnect the runtime. (It unmounts Drive: to run more cells afterwards, run the first cell again.)

In [ ]:
try:
    from google.colab import drive
    drive.flush_and_unmount()
    print("All files are written to Google Drive. You can disconnect the runtime now.")
except ImportError:
    print("Not on Colab: nothing to flush.")